In [ ]:
# Import relevant packages
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from tableone import TableOne
import copy

# Set up the root directory for imports
import pyrootutils
root = pyrootutils.setup_root(
    search_from=os.path.abspath(''),
    indicator=[".git"],
    pythonpath=True,
    dotenv=True,
)

from utils.fns_logistic_regression import LogisticAnalysisOrchestrator 
import forestplot as fp
import re

In [ ]:
# --- Temporal Trends with Multivariate Logistic Regression --
# --- DEFINE VARIABLES ---
df_patient_profiles = pd.read_csv('imputed_patient_profile_analysis.csv')

print(df_patient_profiles.columns)
print(len(df_patient_profiles))

df_patient_profiles.head()

# Prepare data

In [ ]:
meds_cols = ['startdatekey', 'medicationpharmaceuticalsubclass', 'orderedbyprovidertype', 'orderedbyproviderprimaryspecialty']
df_input_profiles = df_patient_profiles.drop(meds_cols, axis=1)
print(len(df_input_profiles))
df_input_profiles = df_input_profiles[df_input_profiles['startdatekey_firstLBP']>20120000]
print(len(df_input_profiles))

categories = [x for x in df_input_profiles.columns if x != 'patientdurablekey']
continuous = ['age_firstLBP', 'startdatekey_firstLBP']
print(categories)

# Load relevant columns from your dataset
df = df_patient_profiles.copy()
df = df[df['startdatekey_firstLBP']>20120000]
df['primaryinsurance'] = df['primaryinsurance'].replace({'0': 'other'})

# Create quartile bins
df['age_group'] = pd.qcut(df['age_firstLBP'], q=3, labels=['18-40', '41-60', '61-90'])
# Print bin edges
quartile_bins = pd.qcut(df['age_firstLBP'], q=3)
print(quartile_bins.unique())

# Ensure LBP_startdatekey is in datetime format
df['startdatekey_firstLBP'] = df['startdatekey_firstLBP'].astype(str)
df['LBP_date'] = pd.to_datetime(df['startdatekey_firstLBP'], format='%Y%m%d', errors='coerce')

# Assign each row to a quarter (start of quarter)
df['quarter'] = df['LBP_date'].dt.to_period('Q').dt.start_time

# Sort and create a time index
df = df.sort_values("quarter").reset_index(drop=True)
df['quarter_num'] = df['quarter'].rank(method='dense').astype(int) - 1  # 0-indexed

# Define policy cutoffs (change points)
#DYNAP
cutoff1 = pd.to_datetime("2015-04-01") # updated 3/15/2025
cutoff2 = pd.to_datetime("2018-10-01") # updated 11/4/2022

# Segment indicators
df['segment_2015Q2'] = (df['quarter'] >= cutoff1).astype(int)
df['segment_2018Q4'] = (df['quarter'] >= cutoff2).astype(int)
df['timebin'] = df['segment_2015Q2'] + df['segment_2018Q4']
# Map values
df['timebin'] = df['timebin'].map({0: 'A', 1: 'B', 2: 'C'})

print(len(df))
df

In [ ]:
df['primaryinsurance'] = df['primaryinsurance'].replace('HMO/POS/EPO','PPO')
print(df['primaryinsurance'].value_counts())

df['lbplaterality'] = df['lbplaterality'].replace(2,1) #unilateral
df['lbplaterality'] = df['lbplaterality'].replace(3,2) #bilateral
print(df['lbpduration'].value_counts())

df_prepped = pd.copy(df,deep=True)

# Descriptive Stats

In [ ]:
# --- Helper Functions ---
def calculate_opioid_rates(df_combined, y_col = 'guideline_period'):
    """Compute opioid prescription rates by guideline period."""
    return {
        period: round(group['opioid'].mean() * 100, 1)
        for period, group in df_combined.groupby(y_col)
    }

def plot_opioid_trends(opioid_rates, output_file='opioid_prescription_trend.png'):
    """Plots opioid prescription rates across guideline periods."""
    plt.figure(figsize=(10, 6))
    plt.bar(opioid_rates.keys(), opioid_rates.values(), color='skyblue')
    plt.ylabel('Opioid Prescription Rate (%)')
    plt.xlabel('CDC Guideline Period')
    plt.title('Opioid Prescription Rates for Lower Back Pain by CDC Guideline Period')
    plt.ylim(0, max(opioid_rates.values()) * 1.2)
    plt.xticks(rotation=45)
    plt.tight_layout()
    #plt.savefig(output_file)
    #plt.close()

In [ ]:
# --- Main Functions ---

# Define variables for TableOne
categorical_vars = ['age_group', 'sex', 'obesity', 'preferredlanguage', 'raceethnicity', 'smokingstatus', 
                'socialsupport', 'primaryinsurance', 'anxiety', 'depression', 
                'lbpduration', 'lbplaterality', 'radiculopathy', 'numbnesstingling', 'sciatica', 'listhesis',
                'facetjointarthropathy', 'scoliosis', 'discpathology', 'spinalstenosis', 'sacroiliacjoint',
                'osteoarthritisososteoarthritis', 'osteopeniaosteoporosis', 'bowelbladder', 'diabetes', 'MR', 'XR', 'CT']
continuous_vars = ['age_firstLBP', 'startdatekey_firstLBP']

### All LBP patients
df_combined = copy.deepcopy(df_prepped)

# Create TableOne
table = TableOne(
    df_combined,
    columns=continuous_vars + categorical_vars,
    categorical=categorical_vars,
    groupby='timebin',
    pval=True,
    pval_adjust='bonferroni',
    label_suffix=True
)

# Save table
table.to_csv('table1_demographic_characteristics_with_changepoints4.csv')
# Display TableOne
table

### Opioid LBP patients
df_opioids = copy.deepcopy(df)
df_opioids = df_opioids[df_opioids['opioid']==1]

# Create TableOne
table = TableOne(
    df_opioids,
    columns=continuous_vars + categorical_vars,
    categorical=categorical_vars,
    groupby='timebin',
    pval=True,
    pval_adjust='bonferroni',
    label_suffix=True
)

# Save table
table.to_csv('table1_demographic_characteristics_with_changepoints_opioids.csv')
# Display TableOne
table

In [ ]:
# Compute and print opioid rates
opioid_rates = calculate_opioid_rates(df_combined, 'timebin')
print("\nOpioid Prescription Rates:")
for period, rate in opioid_rates.items():
    print(f"{period}: {rate:.1f}%")

# Plot
plot_opioid_trends(opioid_rates)

## Temporal Opioids (time A,B,C) percent occurrence and ORs

In [ ]:
df_univariate = df_prepped[['timebin','opioid']]
df_univariate['timebin'] = pd.Categorical(df_univariate['timebin'], categories=['A', 'B', 'C'], ordered=True)
df_univariate.value_counts()

In [ ]:
odds_a = 569/3898
odds_b = 622/6813
odds_c = 886/19152

OR_b_vs_a = odds_b/odds_a
OR_c_vs_a = odds_c/odds_a
OR_c_vs_b = odds_c/odds_b
print(OR_b_vs_a, OR_c_vs_a, OR_c_vs_b)

ci_b_vs_a = np.sqrt(1/569+1/3898+1/622+1/6813)
ci_c_vs_a = np.sqrt(1/569+1/3898+1/886+1/19152)
ci_c_vs_b = np.sqrt(1/622+1/6813+1/886+1/19152)
print(ci_b_vs_a, ci_c_vs_a, ci_c_vs_b)

print(np.round(OR_b_vs_a, 4), OR_b_vs_a+ci_b_vs_a, OR_b_vs_a-ci_b_vs_a)
print(np.round(OR_c_vs_a, 4), OR_c_vs_a+ci_c_vs_a, OR_c_vs_a-ci_c_vs_a)
print(np.round(OR_c_vs_b, 4), OR_c_vs_b+ci_c_vs_b, OR_c_vs_b-ci_c_vs_b)



In [ ]:
percent_a = 569/(569+3898)
percent_b = 622/(622+6813)
percent_c = 886/(886+19152)

print(percent_a, percent_b, percent_c)

# Multivariate Logistic Regression Interaction Analysis

In [ ]:
# --- Helper Functions ---
def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    """Convert appropriate columns to categorical and order timebin."""
    df['timebin'] = pd.Categorical(df['timebin'], categories=['A', 'B', 'C'], ordered=True)
    df['raceethnicity'] = pd.Categorical(df['raceethnicity'], categories=['white', 'latinx', 'asian', 'black or african american', 'other'], ordered=True)
    df['primaryinsurance'] = pd.Categorical(df['primaryinsurance'], categories=['PPO', 'medicare', 'medicaid', 'other'], ordered=True)
    df['smokingstatus'] = pd.Categorical(df['smokingstatus'], categories=['never', 'former', 'smoker'], ordered=True)
    df['lbpduration'] = pd.Categorical(df['lbpduration'], categories=[0, 1, 2], ordered=True)
    df['lbplaterality'] = pd.Categorical(df['lbplaterality'], categories=[0, 1, 2], ordered=True)
    categorical_vars = ['sex', 'preferredlanguage']
    for col in categorical_vars:
        if col in df.columns:
            df[col] = df[col].astype('category')
    return df

# --- MAIN ---
# def main():

df_clean = prepare_data(df_prepped)

# Define analysis parameters
predictor_vars = [
    'age_group', 'sex', 'obesity', 'preferredlanguage', 'raceethnicity', 'smokingstatus', 
    'socialsupport', 'primaryinsurance', 'anxiety', 'depression', 
    'lbpduration', 'lbplaterality', 'radiculopathy', 'numbnesstingling', 'sciatica', 'listhesis',
    'facetjointarthropathy', 'scoliosis', 'discpathology', 'spinalstenosis', 'sacroiliacjoint',
    'osteoarthritisososteoarthritis', 'osteopeniaosteoporosis', 'bowelbladder', 'MR', 'XR', 'CT'
] #'diabetes',
                    
confounders = ['age_firstLBP', 'sex', 'obesity', 'raceethnicity']

# Initialize orchestrator
orchestrator = LogisticAnalysisOrchestrator(decimals=4)

# Run analyses (assuming df_clean is available)
results, interaction_results, interaction_pvals, formatted_OR_results, formatted_interaction_results = orchestrator.analyze_predictors(df_clean, predictor_vars, confounders)
stratified_results = orchestrator.analyze_stratified_predictors(df_clean, predictor_vars, confounders)

print("Analysis framework ready for use!")

# if __name__ == "__main__":
#     main()

In [ ]:
sig_idx = np.where(interaction_pvals < 0.05)[0]
for idx in sig_idx:
    print(predictor_vars[idx])

In [ ]:
results.to_csv('results_interaction_ORs_20250606.csv', index=False)

formatted_results = pd.concat([formatted_OR_results, stratified_results], axis=1)
formatted_results.to_csv('results_interaction_ORs_stratified_ORs_20250610_formatted.csv', index=False)

interaction_results.to_csv('results_interaction_metrics_20250610.csv', index=False)
formatted_interaction_results.to_csv('results_interaction_metrics_20250610_formatted.csv', index=False)

results[0:30]


# Forest Plots

In [ ]:
# --- Helper Functions ---
def extract_time_period(key):
    """Extract time period from the key column"""
    if '[T.B]' in key:
        return 'Time B'
    elif '[T.C]' in key:
        return 'Time C'
    else:
        return 'Time A'  # Default to baseline

# Function to prepare data for forest plot
def prepare_forestplot_data(data, time_period_name):
    """Prepare data for forest plot with required column names"""
    # Split data by time periods
    plot_data = data[data['time_period'] == time_period_name].copy().reset_index(drop=True)

    # Clean up labels for better readability
    plot_data['label'] = (plot_data['key']
                          .str.replace('OR_', '')
                          .str.replace('timebin[T.B]:', 'vs ') # for multiexposure B:insurance 
                          .str.replace('timebin[T.C]:', 'vs ')
                          .str.replace('timebin[T.B]', 'baseline') # for single time exposure B (white at time B vs time A) 
                          .str.replace('timebin[T.C]', 'baseline')
                          .str.replace('[T.', ' (') # for categorical insurance (private)
                          .str.replace(']', ')') # for categorical insurance (private)
        )    
    return plot_data

def add_reference_ORs(df):
    """
    Add Time A baseline rows at the beginning of each predictor's section
    while preserving the original row order within each predictor group
    """
    def _add_baseline_to_group(group):
        """Helper function to add baseline row to each predictor group"""
        predictor = group['predictor'].iloc[0]
        
        # Create the baseline row for this predictor
        baseline_row = pd.DataFrame([{
            'predictor': predictor,
            'key': 'OR_timebin[T.A]',
            'metric': 1,
            'ci_lower': 1,
            'ci_upper': 1,
            'p_value': 0,
            'time_period': 'Time A',
            'label': 'baseline'
        }])
        
        # Concatenate baseline row with the group
        return pd.concat([baseline_row, group], ignore_index=True)
    
    # Create a copy of the dataframe to avoid modifying the original
    df_copy = df.copy()

    # Group by predictor and apply the function to each group
    result = df_copy.groupby('predictor', sort=False).apply(_add_baseline_to_group).reset_index(drop=True)
    
    return result

# Function to create forest plot
def create_forest_plot(data, metric_type = 'OR'):
    """Create forest plot for given data"""
    if len(data) == 0:
        print(f"No data available for forest plot")
        return None
    if metric_type not in ["OR", "ROR", "RERI"]:
        raise ValueError(f"metric_type must be one of or, ror, reri but got '{metric_type}'")
        return None

    if metric_type == 'RERI':
        forest_plot_params = {
                "xlabel":"RERI (95% CI)",    # x-axis label
                "modellabels":["Time C", "Time B", "Time A"],
                "mcolor":["#147301FF", "#65A0DC", "#CC6677"],
                # Vertical reference line parameters:
                "xline": 0,                    # Position of vertical reference line
                "xlim": (-1.5, 1.5),              # (min, max) values * MUST BE NOTED IN THE CAPTION
        }
    elif metric_type == 'ROR':
        forest_plot_params = {
                "xlabel":"ROR (95% CI)",    # x-axis label
                "modellabels":["Time C", "Time B", "Time A"],
                "mcolor":["#147301FF", "#65A0DC", "#CC6677"],
                # Vertical reference line parameters:
                "xline": 1,                    # Position of vertical reference line
                "xlim": (0, 2.5),              # (min, max) values * MUST BE NOTED IN THE CAPTION
        }
    else:
        forest_plot_params = {
                "xlabel":"Odds Ratio (95% CI)",    # x-axis label
                "modellabels":["Time C", "Time B", "Time A"],
                "mcolor":["#77CC66", "#4477AA", "#CC6677"],
                # Vertical reference line parameters:
                "xline": 1,                    # Position of vertical reference line
                "xlim": (0, 2.5),              # (min, max) values * MUST BE NOTED IN THE CAPTION
                }
        
    # Calculate appropriate figure size based on number of rows
    height = max(12, len(data) * 0.1 )  # At least 0.4 inches per row plus margin
    figsize = (5, height)
    figsize = (7.5, 8)
    
    # Create forest plot
    ax = fp.mforestplot(
            dataframe=data,
            estimate="metric",              # odds ratio
            ll="ci_lower", hl="ci_upper",   # confidence intervals
            varlabel="label",        # variable label 
            capitalize=False,        # don't capitalize labels
            model_col="time_period",
            
            groupvar="predictor",    # grouping variable
            color_alt_rows=True,
            sort=False,              # sort within groups
            #group_order=data['predictor'].unique().tolist(),
            pval="p_value",  # Column of p-value to be reported on right
            ylabel="Predictors",
            title="Odds Ratio Comparison to Patients at Time A with the Baseline Characteristic",
            figsize=figsize, 
            markersize=30,
            offset=0.35,  
            fontsize=12,
            grouplab_size=14,
            xlinestyle=(0, (10, 5)),    # Line style (long dash)
            xlinecolor="#808080",       # Line color (gray)
            xlinewidth=1.5,             # Line width
            **forest_plot_params
        )

    # Manual label positioning to ensure visibility
    # This fixes the forestplot package's positioning issues with large figures
    ax.tick_params(axis='y', which='major', pad=20)
    for label in ax.get_yticklabels():
        label.set_horizontalalignment('right')
        label.set_x(-0.02)  # Position labels close to plot area

    # Enable subplot configuration tool
    plt.subplots_adjust(left=0.652, right=0.9, top=1, bottom=0.176)
    
    # Customize appearance
    plt.tight_layout()
    return ax


### Odds Ratios Forest Plot

In [ ]:
clean_df = pd.read_csv('results_interaction_ORs_20250606.csv')

print('Check for large confidence intervals and remove them')
print(clean_df[clean_df['ci_upper']>10])
# clean_df = clean_df[clean_df['predictor']!='lbplaterality']
# clean_df = clean_df[clean_df['predictor']!='osteopeniaosteoporosis']
print(max(clean_df['ci_upper']))

# Add time period column
clean_df['time_period'] = clean_df['key'].apply(extract_time_period)

# Prepare data for each time period
time_a_plot = prepare_forestplot_data(clean_df, 'Time A')
time_a_plot['label'] = 'vs ' + time_a_plot['label']
time_a_plot = add_reference_ORs(time_a_plot)
time_b_plot = prepare_forestplot_data(clean_df, 'Time B')
time_c_plot = prepare_forestplot_data(clean_df, 'Time C')

print(f"Time A: {len(time_a_plot)} observations")
print(f"Time B: {len(time_b_plot)} observations")
print(f"Time C: {len(time_c_plot)} observations")

clean_df = pd.concat((time_a_plot, time_b_plot, time_c_plot),axis=0)
clean_df['label'] = clean_df['label'].apply(lambda x: 'vs ' + re.search(r'\((.*?)\)', x).group(1) if '(' in x and ')' in x else x)

In [ ]:
# Filter to include age_group, sex, and 10 additional clinically relevant variables
psychosocial = ['age_group', 'sex', 'obesity', 'preferredlanguage', 'raceethnicity', #lbpdate
                'smokingstatus', 'socialsupport', 'primaryinsurance', 'anxiety', 'depression']
diagnoses = ['lbpduration', 'radiculopathy', 'numbnesstingling', 'sciatica', 'listhesis', 'facetjointarthropathy', 'scoliosis', #lbplaterality
                'discpathology', 'spinalstenosis', 'sacroiliacjoint', 'osteoarthritisososteoarthritis', 'bowelbladder', #osteopeniaosteoporosis
                'MR', 'XR', 'CT'] #'diabetes'

for selected_predictors in [psychosocial, diagnoses]:

    plot_df = clean_df[clean_df['predictor'].isin(selected_predictors)].copy()

    # Create forest plots for each predictor
    print("Creating forest plots for temporal opioid prescription analysis...")

    # Convert 'predictor' column to categorical to preserve its current order
    plot_df['predictor'] = pd.Categorical(plot_df['predictor'], categories=plot_df['predictor'].unique(), ordered=True)
    plot_df['time_period'] = pd.Categorical(plot_df['time_period'], categories=['Time A', 'Time B', 'Time C'], ordered=True)

    # Then sort by predictor (preserving order), then time_period and label ascending
    df_sorted = plot_df.sort_values(by=['predictor', 'time_period', 'label'], ascending=[True, True, True])
    df_sorted['label'] = pd.Categorical(df_sorted['label'], categories=df_sorted['label'].unique(), ordered=True)

    ax = create_forest_plot(df_sorted)
    plt.show()
    # plt.savefig('results/summary_forest_plot.png', dpi=300, bbox_inches='tight')


### ROR and RERI Forest Plots

In [ ]:
interaction_term_results = pd.read_csv('results_interaction_metrics_20250610.csv')
clean_df = interaction_term_results[['predictor', 'key', 'metric', 'ci_lower', 'ci_upper', 'p_value']]

print('Check for large confidence intervals and remove them')
print(clean_df[clean_df['ci_upper']>10])

interaction_predictors = ['age_group', 'raceethnicity', 'smokingstatus', 'socialsupport', 
                          'primaryinsurance', 'MR', 'XR', 'CT']
# not_significant_interaction_predictors = ['lbpduration','numbnesstingling', 'sciatica', 'listhesis', 
# 'facetjointarthropathy', 'sacroiliacjoint', 'osteoarthritisososteoarthritis']
# removed_predictors = ['diabetes', 'lbplaterality', 'osteopeniaosteoporosis'] # removed
clean_df = clean_df[clean_df['predictor'].isin(interaction_predictors)]
print(max(clean_df['ci_upper']))

# Add time period column
clean_df['time_period'] = clean_df['key'].apply(extract_time_period)

# Prepare data for each time period
time_b_plot = prepare_forestplot_data(clean_df, 'Time B')
time_c_plot = prepare_forestplot_data(clean_df, 'Time C')
print(f"Time B: {len(time_b_plot)} observations")
print(f"Time C: {len(time_c_plot)} observations")

clean_df = pd.concat((time_b_plot, time_c_plot),axis=0)
clean_df['label'] = clean_df['key'].apply(
    lambda x: 'vs ' + (
        re.search(r':.*?\[T\.?([^\]]+)\]', x).group(1) if re.search(r':.*?\[T\.?([^\]]+)\]', x)
        else x.split(':')[1].strip()
    ) if ':' in x else x
)
#clean_df 

In [ ]:
for interaction_term in ['ROR', 'RERI']:

    plot_df = clean_df[clean_df['key'].str.contains(interaction_term)].copy()

    # Create forest plots for each predictor
    print("Creating forest plots for temporal opioid prescription analysis...")

    # Convert 'predictor' column to categorical to preserve its current order
    plot_df['predictor'] = pd.Categorical(plot_df['predictor'], categories=plot_df['predictor'].unique(), ordered=True)
    plot_df['time_period'] = pd.Categorical(plot_df['time_period'], categories=['Time B', 'Time C'], ordered=True)

    # Then sort by predictor (preserving order), then time_period and label ascending
    df_sorted = plot_df.sort_values(by=['predictor', 'time_period', 'label'], ascending=[True, True, True])
    df_sorted['label'] = pd.Categorical(df_sorted['label'], categories=df_sorted['label'].unique(), ordered=True)

    ax = create_forest_plot(df_sorted, metric_type=interaction_term)
    plt.show()
    # plt.savefig('results/summary_forest_plot.png', dpi=300, bbox_inches='tight')
